# 🫀 퀘스트 46 · Q4-D — **보정기를 바꾸면 π̂ 의존이 사라지는가**

| | **MedKOS / `notebooks/quest46_q4d_calibrator_shift_equivariance.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q4c_burden_train_vs_test`(`20260805T0212`) · `quest46_q4b_burden_feature_v2` |
| 성격 | **원인 제거 + 대가 측정** — 진단이 아니라 **처방**을 고치는 런 |

## Q4-C 가 특정한 것

`add` 모드에서 burden 은 레코드 내 상수이고 선형으로 들어간다. 그래서 **보정 전**
테스트시 burden 은 레코드 내 판별에 **원리적으로 기여하지 않는다** — 실측으로
`TT` 와 `TS` 의 보정 전 매크로가 **0.577231 로 완전히 같았고 순위가 56/56 일치**했다.

그런데 **보정 후**엔 갈라졌다:

```
보정 전  0.577231
보정 후  TT 0.5698(−0.0074) · TE 0.5663(−0.0109) · TS 0.5353(−0.0419)
         ↑ 손실이 b̂ 오차 크기를 그대로 따라간다
```

**범인은 등장성 보정이다.** 계단함수라 상수 시프트가 예측을 다른 계단으로 밀어
**동점 구조**를 바꾼다.

## ★★★ 그런데 이건 구성으로 고칠 수 있다

Platt 보정은 `cal(s) = σ(a·s + b)` 이므로

$$\mathrm{logit}(\mathrm{cal}(s)) = a\,s + b$$

**보정된 로짓이 원점수의 아핀 변환**이다. 그러면 테스트시 burden 의 상수 시프트
`s → s + c` 가 `a·s + b → a·s + b + a·c` 로 **상수 시프트인 채 보존**되고,
**레코드 내 매크로가 정확히 불변**이 된다. 근사가 아니라 **항등**이다.

⚠️ **단, 확률로 왕복하면 깨진다.** `logit(clip(cal(s), 1e-6, 1−1e-6))` 는 포화
구간에서 clip 에 걸려 항등이 무너진다(실측 시프트 후 차이의 SD 5.86e-14, 포화가
심하면 훨씬 커진다). 그래서 이 런의 보정기는 **보정된 로짓을 직접** 반환한다 —
Platt 은 해석적으로 `a·s+b`, 등장성은 `logit(clip(·))`.

## 그래서 무엇을 묻나

**바꾸는 게 이득인가, 아니면 대가가 더 큰가.** 등장성은 fold 간 척도를 더 잘 맞춘다
(그래서 Q3 이 골랐다). Platt 은 안 새지만 보정이 거칠 수 있다. 그 **거래**를 잰다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **G0** | 코호트 · A 항등 (보정기별) | 구성 항등. 깨지면 **중단** |
| **G1 ★★★ 자** | **Platt 하에서** `TT` ≡ `TE` ≡ `TS` 매크로가 **정확히** 같은가 | 허용 1e-12. 깨지면 **중단** |
| **G2 ★★★ 주 관문** | 보정기별 **배포 가능판** `TE − A_em` (매크로 · 교차레코드) | max(0, 측정된 영점 상단) 초과 |
| **G3 ★★ 대가** | 보정기 교체의 값 — `TE@platt − TE@iso` (매크로 · 교차 · ECE) | 관문 아님. **거래를 보고** |
| **G4** | **레코드별 누출** — 등장성에서 어디가 새는가 | 관문 아님 |
| **G5 ★★** | **전역 이득의 정체** — 유병률 가중인가 판별력인가 | 관문 아님. Q3 소급 재해석 |
| **G6** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **G1 ✅ · G2(platt) ✅** → **π̂ 의존 없는 배포 처방**이 선다. 층② 종결
- **G1 ✅ · G2(platt) ❌/미결** → 누출은 없앴지만 **보정 품질 대가가 더 크다** → 등장성 유지
- **G1 ❌** → 구현 오류. **중단**

## Q4-C 설계 정정 (함께 반영)

1. **테스트 반사실을 배포판으로** — Q4-C 의 F4 는 `TT − TS`(derangement, **적대적**)를
   썼고 그래서 테스트 몫이 54% 로 보였다. 배포 반사실 `TT − TE` 로는 **5%** 다.
   `TS` 는 **적대적 상한**으로만 표기한다
2. **상호작용을 먼저 본다** — Q4-C 는 상호작용이 **+0.0236(합계의 37%)** 인데
   한 경로의 분해를 판정에 썼다. 크면 **분해하지 않는다**
3. **보정 전을 네 칸 전부** — Q4-C 는 `TT`·`TS` 만 쟀다
4. **N_PERM 5 → 10** — Q4-C 의 F3 는 **영점 CI 폭(0.0389)이 효과(0.0167)보다 커서**
   영점의 정밀도가 병목이었다
5. **요약이 「해석 불가」로 선언한 수를 인용하지 않는다**(Q4-C 가 F3 필요표본 625 를
   인용했다 · R38 ⑦)

⚠️ **새 데이터 0** — `svdb_data5.npz` 만.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def _rank_avg(v):
    v = np.asarray(v, float); o = v.argsort()
    r = np.empty(len(v), float); r[o] = np.arange(len(v), dtype=float)
    for u in np.unique(v):
        m = v == u
        if m.sum() > 1:
            r[m] = r[m].mean()
    return r

def spearman(a, b):
    ra, rb = _rank_avg(a), _rank_avg(b)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    """★★ `eff` 는 **관문 문턱과의 거리**다(영점 평균이 아니라) — Q4-B 오류의 정정."""
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def ece_of(p, y, nbin=15):
    """보정 품질 — Platt 로 바꾸는 **대가**가 여기서 보인다."""
    p = np.asarray(p, float); y = np.asarray(y, float)
    edges = np.linspace(0.0, 1.0, nbin + 1); e = 0.0
    for i in range(nbin):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < nbin - 1 else p <= edges[i + 1])
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def derangement(n, rng):
    for _ in range(1000):
        p = rng.permutation(n)
        if not np.any(p == np.arange(n)):
            return p
    return np.roll(np.arange(n), 1)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
FS = 360
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수 (SMOKE 가 절대 안 건드린다)
TOL_IDENT = 1e-12
DEV_EVERY = 4

# ── 비용 손잡이
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 10     # ★ Q4-C 5 → 10 (영점 정밀도가 F3 의 병목이었다)

# ★★★ 이 런의 축은 **보정기**다.
CALS = ("iso", "platt")
PRIMARY_CAL = "platt"              # 시프트 등변인 쪽이 처방 후보다
BSPEC = {"TT": ("true", "true"), "TE": ("true", "em"), "TS": ("true", "shuf"),
         "ST": ("shuf", "true"),  "SS": ("shuf", "shuf")}
ARMS = ("raw", "A_oracle", "A_em") + tuple(BSPEC)
# ★★ 배포 가능판끼리가 주 관문이다 — 오라클은 상한이지 방법이 아니다.
MAIN = ("A_em", "TE")
PRIMARY = ("macro", "xrec")
REPORT_ONLY = ("pooled", "pooled_bal")
READ_ORDER = ("G0", "G1", "G2", "G3", "G4", "G5", "G6")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(   # Q4-C(`20260805T0212`) 실측 — 같은 코호트라 **재현 앵커**다
    n_ok=56, dom_rec=48, dom_share=0.152,
    q4c_pre_macro=0.577231, q4c_pre_ident=0.0, q4c_pre_rank="56/56",
    q4c_macro=dict(raw=0.5117, A_oracle=0.5117, A_em=0.5117, TT=0.5698,
                   TE=0.5663, TS=0.5353, ST=0.5170, SS=0.5061),
    q4c_xrec=dict(raw=0.8829, A_oracle=0.8679, A_em=0.7501, TT=0.8845,
                  TE=0.8980, TS=0.8510, ST=0.8838, SS=0.8785),
    q4c_pool=dict(raw=0.2097, A_oracle=0.4495, A_em=0.1254, TT=0.5268,
                  TE=0.3680, TS=0.2167, ST=0.2115, SS=0.2083),
    q4c_F2=0.0582, q4c_F5m=0.0546, q4c_F5x=0.1479,
    q4c_iso_leak=dict(TT=-0.0074, TE=-0.0109, TS=-0.0419),   # 보정 전 대비 손실
    q4c_interaction=0.0236, q4c_total=0.0638,
    q4c_test_adv=0.0346, q4c_test_deploy=0.0035,             # 적대적 vs 배포 반사실
    q4c_dom48=dict(raw=0.6169, TT=0.8034, TE=0.8474, TS=0.8321, ST=0.6162, SS=0.6178))

RULE_CHECK = {
    "R11 매크로":       "매크로·교차레코드가 공동 주 지표. 전역은 **접힌 채**로 둔다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "LORO 안에서 기저·보정을 **held-out 레코드를 빼고** 적합",
    "R26 / R38 ②":      "★ 영점을 rep×레코드로 측정. 못 쟀으면 **안 읽는다**. N_PERM 10",
    "R29 ② 분기 금지":   "G0 · G1 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ③ 대조":       "`shuf` 는 같은 값 집합, 대응만 깨진다 — 단 **적대적 상한**으로만 읽는다",
    "R35 ① 자 먼저":    "★★★ **G1 이 이 런의 자다** — Platt 하 항등을 먼저 세운다",
    "R36 ① 접기/닫기":  "전역은 **접힌 채**(상한 +0.2008 · 필요 356 · 가용 56)",
    "R36 ② 선택 편의":  "★ 주 관문을 **배포 가능판끼리**로 고정 — 오라클은 방법이 아니다",
    "R40 ① λ ≠ 타당성":  "★★ 보정 품질(ECE)이 좋다고 판별이 좋은 게 아니다 — **둘 다** 잰다",
    "R40 ② 같은 통계":  "★ 필요표본을 **관문 문턱 기준**으로",
    "R41 ② 0 근처":     "★ 상호작용이 크면 **분해하지 않는다**(Q4-C 정정)",
}

CONFIG = dict(
    exp="quest46_q4d_calibrator_shift_equivariance", quest="ailab-2026-0046",
    step="calibrator-shift-equivariance",
    parent_exp=["quest46_q4c_burden_train_vs_test", "quest46_q4b_burden_feature_v2"],
    purpose=("**원인 제거 + 대가 측정.** Q4-C(`20260805T0212`)가 원인을 특정했다 — "
             "`add` 모드에서 burden 은 레코드 내 상수·선형이므로 **보정 전** 테스트시 "
             "burden 은 레코드 내 판별에 **원리적으로 기여하지 않는다**(실측 `TT`≡`TS` "
             "매크로 0.577231 · 순위 56/56 일치). 그런데 보정 후엔 갈라졌고"
             "(TT −0.0074 · TE −0.0109 · TS −0.0419) **손실이 b̂ 오차 크기를 그대로 "
             "따라간다** → 범인은 **등장성 보정의 계단 구조**다. "
             "★★★ 이건 **구성으로 고칠 수 있다**: Platt 이면 `logit(σ(as+b)) = as+b` 라 "
             "테스트시 burden 의 상수 시프트가 **상수 시프트인 채 보존**되어 매크로가 "
             "**정확히 불변**이 된다(근사가 아니라 항등). ⚠️ 단 확률로 왕복하면 clip 에 "
             "걸려 깨지므로(실측 시프트 후 SD 5.86e-14) 이 런의 보정기는 **보정된 로짓을 "
             "직접** 반환한다. 그래서 묻는 것은 **「바꾸는 게 이득인가」** 다 — 등장성은 "
             "fold 간 척도를 더 잘 맞추고(그래서 Q3 이 골랐다) Platt 은 안 새지만 보정이 "
             "거칠 수 있다. 그 **거래**를 잰다."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · 새 데이터 0)",
    cals=list(CALS), primary_cal=PRIMARY_CAL, arms=list(ARMS),
    bspec={k: list(v) for k, v in BSPEC.items()}, main=list(MAIN),
    primary=list(PRIMARY), report_only=list(REPORT_ONLY), read_order=READ_ORDER,
    dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "G0": "코호트 + A 항등(보정기별). A 는 **보정 뒤** 상수 로짓 시프트라 매크로가 "
              "raw 와 정확히 같아야 한다 — 보정기와 무관하게. 깨지면 **중단**",
        "G1": "★★★ **이 런의 자(R35 ①)** — **Platt 하에서** `TT`·`TE`·`TS` 의 매크로가 "
              "**정확히 같아야** 한다(테스트시 burden 만 다른 세 팔). 아핀 보정은 상수 "
              "시프트를 상수 시프트로 보존하기 때문이고, **근사가 아니라 항등**이다. "
              "등장성에서는 갈라지며 그 차가 **누출의 실측치**다",
        "G2": "★★★ **주 관문 — 배포 가능판끼리** `TE − A_em` 을 **보정기별로**. "
              "오라클은 상한이지 방법이 아니다(Q4-C: `A_em` 전역 0.1254 < raw 0.2097). "
              "매크로·교차레코드 둘 다 max(0, 측정된 영점 상단) 초과가 기준",
        "G3": "★★ **교체의 대가(관문 아님)** — `TE@platt − TE@iso` 를 매크로·교차레코드·"
              "**ECE** 로. R40 ①: 보정이 좋다고 판별이 좋은 게 아니므로 **둘 다** 본다. "
              "누출을 없앤 값이 척도 정렬의 손해보다 큰가",
        "G4": "**레코드별 누출(관문 아님)** — 등장성에서 레코드마다 |매크로(TT)−매크로(TS)| 를 "
              "재고, 시프트 크기·유병률·보정기 계단 수와 어떻게 붙는지 본다. Q4-C 에서 "
              "지배 레코드 48 은 누출이 사실상 0 이었다(TS 0.8321 vs TT 0.8034)",
        "G5": "★★ **전역 이득의 정체(관문 아님)** — Q4-C 에서 `A_oracle − raw` 가 전역 "
              "**+0.2398** 인데 교차레코드는 **−0.0150** 이었다. 전역을 **레코드 크기 균등 "
              "가중**으로 다시 재서(`pooled_bal`) 「유병률 가중」의 몫을 분리한다. "
              "Q3 의 +0.2151 **소급 재해석**이 걸려 있다",
        "G6": "결론 검산표"},
    caveat=("★★★ **G1 은 가설이 아니라 구현 검사다** — Platt 하 항등은 수식에서 따라오므로, "
            "안 서면 코드가 틀린 것이다. 그래서 실패 시 **중단**한다. "
            "★★ **누출을 없애는 것과 좋아지는 것은 다른 말이다**(R40 ①) — Platt 이 π̂ "
            "의존을 0 으로 만들어도 보정이 거칠어 교차레코드가 나빠지면 **순손해**다. "
            "G2·G3 을 **함께** 읽는다. "
            "★ **Q4-C 정정** — 테스트 반사실은 **배포판(`TE`)**이다. `TS`(derangement)는 "
            "**적대적 상한**으로만 표기한다(Q4-C 는 이걸 판정에 써서 테스트 몫을 54% 로 "
            "봤는데 배포 반사실로는 5% 다). 상호작용이 크면 **분해하지 않는다**."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4d_calibrator_shift_equivariance", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-D — 보정기를 바꾸면 π̂ 의존이 사라지는가**")
run.log("  ★★★ Platt: `logit(σ(as+b)) = as+b` → 테스트시 burden 의 상수 시프트가 "
        "**상수 시프트인 채 보존** → 매크로 **정확히 불변**")
run.log("  ★★ 보정기는 **보정된 로짓을 직접** 반환한다 — 확률로 왕복하면 clip 에 걸려 "
        "항등이 깨진다(실측 SD 5.86e-14)")
run.log("  ★★ 묻는 것은 「없앨 수 있나」가 아니라 **「바꾸는 게 이득인가」** 다 — "
        "등장성은 fold 간 척도를 더 잘 맞춘다(R40 ①)")
run.log(f"  ★ 주 관문은 **배포 가능판끼리** `{MAIN[1]} − {MAIN[0]}` · N_PERM {N_PERM}")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【G-0】 코호트 · 보정기 · LORO · 지표 넷
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

run.log("\n" + "=" * 100)
run.log("【G-0】 코호트 · 보정기 · LORO 골격")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
RHY = np.nan_to_num(np.c_[_med - pre,
                          np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                          post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS = {int(r): np.where(RID == r)[0] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
s_all = np.array([int(TT_[IDXS[r]].sum()) for r in REC_OK], float)
DOMINANT = float(s_all.max() / s_all.sum()); DOM_REC = int(REC_OK[int(np.argmax(s_all))])
NRE = len(REC_OK)
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 유병률 "
        f"{min(BURD.values()):.4f}~{max(BURD.values()):.4f} · 지배 지분 {DOMINANT:.3f}")

EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

# ── ★★★ 보정기는 (확률, **보정된 로짓**) 두 함수를 낸다.
#    Platt 의 보정 로짓은 **해석적으로** a·s+b 다 — 확률로 왕복하면 clip 에 걸려
#    시프트 등변성이 깨진다(실측: 시프트 후 차이의 SD 5.86e-14, 포화가 심하면 더 크다).
#    이건 편법이 아니라 **더 정확한 구현**이고, G1 을 항등으로 만드는 자리다.
def make_cal(kind, s, y):
    s = np.asarray(s, float); y = np.asarray(y).astype(int)
    if kind == "iso":
        ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6).fit(s, y.astype(float))
        p_ = lambda v: np.clip(ir.predict(np.asarray(v, float)), EPS, 1 - EPS)
        return p_, (lambda v: logit(p_(v))), int(len(np.unique(ir.predict(s))))
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(s.reshape(-1, 1), y)
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    return (lambda v: 1.0 / (1.0 + np.exp(-(a * np.asarray(v, float) + b))),
            lambda v: a * np.asarray(v, float) + b, 0)

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def loro_B(cal_kind, train_src, test_src, shuf_map, y_override=None):
    """반환: (보정 로짓, 보정 전 점수, 보정 확률, fold 별 계단 수)"""
    out = np.full(len(K), np.nan); pre_ = np.full(len(K), np.nan)
    prob = np.full(len(K), np.nan); steps = {}
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        bmap = BURD if train_src == "true" else shuf_map
        mu = float(np.mean([bmap[r] for r in tr_r]))
        bv = lambda ii: np.array([bmap[int(r)] for r in RID[ii]], float)
        Ftr = np.c_[RHY[tr], bv(tr)]; fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii, b_: lr.decision_function((np.c_[RHY[ii], b_] - fmu) / fsd)
        cp, cl, st = make_cal(cal_kind, sc(dv, bv(dv)), TT_[dv]); steps[held] = st
        pi_tr = float(TT_[dv].mean())
        if test_src == "true":
            bh = BURD[held]
        elif test_src == "shuf":
            bh = shuf_map[held]
        else:
            bh = em_prior(cp(sc(te, np.full(len(te), mu))), pi_tr)
        s_te = sc(te, np.full(len(te), bh))
        pre_[te] = s_te; out[te] = cl(s_te)
    # ★ ECE 는 **최종 로짓**에서 낸 확률로 잰다 — 배포되는 게 그것이다
    prob = 1.0 / (1.0 + np.exp(-np.clip(out, -60, 60)))
    return out, pre_, prob, steps

def loro_A(cal_kind, kind, y_override=None):
    out = np.full(len(K), np.nan); prob = np.full(len(K), np.nan)
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        Ftr = RHY[tr]; fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii: lr.decision_function((RHY[ii] - fmu) / fsd)
        cp, cl, _ = make_cal(cal_kind, sc(dv), TT_[dv])
        pi_tr = float(TT_[dv].mean())
        l = cl(sc(te)); p_pre = cp(sc(te))
        if kind == "oracle":
            l = l + (logit(BURD[held]) - logit(pi_tr))
        elif kind == "em":
            l = l + (logit(em_prior(p_pre, pi_tr)) - logit(pi_tr))
        out[te] = l
    # ★ 사전확률 시프트 **뒤**의 확률로 ECE 를 잰다(A 팔이 raw 와 같게 나오면 안 된다)
    prob = 1.0 / (1.0 + np.exp(-np.clip(out, -60, 60)))
    return out, prob

# ── 지표 넷: 매크로 · 교차레코드 · 전역 · **레코드 균등 가중 전역**(G5 용)
WBAL = np.zeros(len(K))
for r in REC_OK:
    WBAL[IDXS[r]] = 1.0 / len(IDXS[r])
FIN = np.zeros(len(K), bool)
for r in REC_OK:
    FIN[IDXS[r]] = True
pooled_of = lambda L: float(average_precision_score(TT_[FIN].astype(int), L[FIN]))
pooled_bal_of = lambda L: float(average_precision_score(TT_[FIN].astype(int), L[FIN],
                                                        sample_weight=WBAL[FIN]))
def per_rec(L):
    d = {}
    for r in REC_OK:
        pos = IDXS[r]; yy = TT_[pos].astype(int)
        if 0 < yy.sum() < len(yy) and np.all(np.isfinite(L[pos])):
            d[r] = float(average_precision_score(yy, L[pos]))
    return d

def xrec_matrix(L):
    P = {r: np.sort(L[IDXS[r]][TT_[IDXS[r]]]) for r in REC_OK}
    N = {r: np.sort(L[IDXS[r]][~TT_[IDXS[r]]]) for r in REC_OK}
    M = np.full((NRE, NRE), np.nan)
    for a, ri in enumerate(REC_OK):
        p = P[ri]
        if not len(p):
            continue
        for b, rj in enumerate(REC_OK):
            if ri == rj or not len(N[rj]):
                continue
            q = N[rj]
            lo = np.searchsorted(q, p, "left"); hi = np.searchsorted(q, p, "right")
            M[a, b] = float((lo + 0.5 * (hi - lo)).sum() / (len(p) * len(q)))
    return M

def xrec_of(M):
    off = ~np.eye(NRE, dtype=bool)
    v = M[off & np.isfinite(M)]
    return float(v.mean()) if len(v) else float("nan")

def xrec_boot(Ms, seed, nb):
    rng = np.random.RandomState(seed); out = {k: [] for k in Ms}
    for _ in range(nb):
        idx = rng.randint(0, NRE, NRE); same = idx[:, None] == idx[None, :]
        for k, M in Ms.items():
            sub = M[np.ix_(idx, idx)]; m = (~same) & np.isfinite(sub)
            out[k].append(float(sub[m].mean()) if m.any() else float("nan"))
    return {k: np.asarray(v, float) for k, v in out.items()}

run.log("  보정기 정의 완료 — `iso`(등장성·계단) · `platt`(아핀·**시프트 등변**)")
CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=NRE, dominant=DOMINANT, dom_rec=DOM_REC)
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【G-A】 전 팔 × 두 보정기 · G0 항등 · ★★★ G1 자
run.log("\n" + "=" * 100)
run.log("【G-A】 실행 · G0(A 항등) · ★★★ G1(Platt 하 `TT`≡`TE`≡`TS`)")
run.log("=" * 100)
T0 = time.time()
perm0 = derangement(NRE, np.random.RandomState(SEED0 + 300))
SHUF = {REC_OK[i]: BURD[REC_OK[perm0[i]]] for i in range(NRE)}

L, PROB, PRECAL, STEPS = {}, {}, {}, {}
for c in CALS:
    L[c], PROB[c], PRECAL[c] = {}, {}, {}
    L[c]["raw"], PROB[c]["raw"] = loro_A(c, "raw")
    for k_ in ("oracle", "em"):
        L[c][f"A_{k_}"], PROB[c][f"A_{k_}"] = loro_A(c, k_)
    for nm, (tr_s, te_s) in BSPEC.items():
        L[c][nm], PRECAL[c][nm], PROB[c][nm], st = loro_B(c, tr_s, te_s, SHUF)
        if nm == "TT":
            STEPS[c] = st
    run.log(f"  ({time.time()-T0:>5.0f}초) 보정기 `{c}` — {len(ARMS)}팔 완료")

MACRO, XREC, XM, POOL, PBAL, ECE = {}, {}, {}, {}, {}, {}
for c in CALS:
    PER_ = {a: per_rec(L[c][a]) for a in ARMS}
    MACRO[c] = {a: float(np.mean(list(PER_[a].values()))) for a in ARMS}
    XM[c] = {a: xrec_matrix(L[c][a]) for a in ARMS}
    XREC[c] = {a: xrec_of(XM[c][a]) for a in ARMS}
    POOL[c] = {a: pooled_of(L[c][a]) for a in ARMS}
    PBAL[c] = {a: pooled_bal_of(L[c][a]) for a in ARMS}
    ECE[c] = {a: ece_of(PROB[c][a][FIN], TT_[FIN]) for a in ARMS}
    globals()[f"PER_{c}"] = PER_
PER = {c: globals()[f"PER_{c}"] for c in CALS}

for c in CALS:
    run.log(f"\n  보정기 `{c}`   {'팔':<12}{'매크로':>10}{'교차':>10}{'전역':>10}"
            f"{'전역(균등)':>12}{'ECE':>9}")
    for a in ARMS:
        run.log(f"  {'':<14}{a:<12}{MACRO[c][a]:>10.4f}{XREC[c][a]:>10.4f}"
                f"{POOL[c][a]:>10.4f}{PBAL[c][a]:>12.4f}{ECE[c][a]:>9.4f}")
run.log(f"\n  (Q4-C 앵커 `iso` — 매크로 raw {REF['q4c_macro']['raw']} · TT "
        f"{REF['q4c_macro']['TT']} · TE {REF['q4c_macro']['TE']} · TS {REF['q4c_macro']['TS']}"
        f" | 교차 A_em {REF['q4c_xrec']['A_em']} · TE {REF['q4c_xrec']['TE']})")

# ── G0 — A 는 **보정 뒤** 상수 로짓 시프트라 매크로가 raw 와 같아야 한다(보정기 무관)
d0 = max(abs(MACRO[c][a] - MACRO[c]["raw"]) for c in CALS for a in ("A_oracle", "A_em"))
run.log(f"\n  G0 — A 팔 매크로 vs raw · 두 보정기 통틀어 max|Δ| = **{d0:.2e}**")
if d0 >= TOL_IDENT:
    raise AssetError(f"G0 실패({d0:.3e}) — A 는 보정 뒤 상수 시프트다(R29 ②)")
g_("G0", "✅ 지지", f"A 팔 매크로가 raw 와 **정확히 같다**({d0:.1e}) — 보정기와 무관하다")

# ── ★★★ G1 — Platt 하에서 테스트시 burden 만 다른 세 팔이 **정확히** 같은가
run.log("\n  ★★★ G1 — 테스트시 burden 만 다른 세 팔(`TT`·`TE`·`TS`)의 매크로")
run.log("     Platt: logit(σ(a·s+b)) = a·s+b → 상수 시프트가 **상수 시프트인 채 보존**")
run.log("     등장성: 계단함수 → 시프트가 **동점 구조**를 바꾼다")
TRIO = ("TT", "TE", "TS")
G1 = {}
for c in CALS:
    vals = [MACRO[c][a] for a in TRIO]
    spread = float(max(vals) - min(vals))
    pre_m = {a: float(np.mean([average_precision_score(TT_[IDXS[r]].astype(int),
                                                       PRECAL[c][a][IDXS[r]])
                               for r in REC_OK])) for a in BSPEC}
    # ★ Q4-C 정정 — 보정 전을 **네 칸 전부** 잰다(Q4-C 는 TT·TS 만 쟀다)
    pre_T = abs(pre_m["TT"] - pre_m["TS"]); pre_S = abs(pre_m["SS"] - pre_m["ST"])
    G1[c] = dict(vals={a: MACRO[c][a] for a in TRIO}, spread=spread, pre=pre_m,
                 pre_ident_T=pre_T, pre_ident_S=pre_S)
    run.log(f"    `{c:<6}` " + " · ".join(f"{a} {MACRO[c][a]:.6f}" for a in TRIO)
            + f"  → 폭 **{spread:.2e}**")
    run.log(f"             보정 전 — 학습 true 행 |TT−TS| {pre_T:.2e} · "
            f"학습 shuf 행 |SS−ST| {pre_S:.2e} (둘 다 0 이어야 한다)")
    if max(pre_T, pre_S) >= TOL_IDENT:
        raise AssetError(f"보정 전 항등 실패(`{c}` {max(pre_T, pre_S):.3e}) — 구현 오류")
if G1[PRIMARY_CAL]["spread"] >= TOL_IDENT:
    raise AssetError(f"G1 실패 — Platt 하 폭 {G1[PRIMARY_CAL]['spread']:.3e} ≥ {TOL_IDENT:.0e}. "
                     "아핀 보정은 상수 시프트를 보존하므로 **수식에서 따라오는 항등**이다. "
                     "안 서면 코드가 틀린 것이므로 아래를 읽지 않는다(R29 ②)")
ISO_LEAK = G1["iso"]["spread"]
g_("G1", "✅ 지지",
   f"**Platt 에서 π̂ 의존이 정확히 0 이다**(폭 {G1[PRIMARY_CAL]['spread']:.1e} < "
   f"{TOL_IDENT:.0e}) — 테스트시 burden 을 오라클·EM·완전셔플 중 무엇으로 줘도 매크로가 "
   f"**같다**. 등장성에서는 폭이 **{ISO_LEAK:.4f}** 로 벌어진다(Q4-C 가 본 그 누출)")
CONFIG["G0"] = dict(ident=float(d0),
                    macro=MACRO, xrec=XREC, pooled=POOL, pooled_bal=PBAL, ece=ECE)
CONFIG["G1"] = dict(per_cal=G1, iso_leak=float(ISO_LEAK), tol=TOL_IDENT)
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【G-B】 ★★★ G2 주 관문(보정기별 배포 가능판) · 영점
run.log("\n" + "=" * 100)
run.log(f"【G-B】 ★★★ G2 — **배포 가능판끼리** `{MAIN[1]} − {MAIN[0]}` (보정기별)")
run.log("=" * 100)
run.log("  오라클은 **상한이지 방법이 아니다** — Q4-C 실측 `A_em` 전역 "
        f"{REF['q4c_pool']['A_em']} < raw {REF['q4c_pool']['raw']}")

T2 = time.time()
BX = {c: xrec_boot({a: XM[c][a] for a in ARMS}, SEED0 + 11, NB_BOOT) for c in CALS}
run.log(f"  ({time.time()-T2:.0f}초) 교차레코드 부트스트랩 {NB_BOOT}회 × {len(CALS)}보정기")

def d_macro(c, a, b, seed):
    ks = [r for r in REC_OK if r in PER[c][a] and r in PER[c][b]]
    m_, lo_, hi_, n_ = boot_pair([PER[c][a][r] for r in ks], [PER[c][b][r] for r in ks],
                                 seed, NB_BOOT)
    return dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))

def d_xrec(c, a, b):
    d = BX[c][b] - BX[c][a]
    lo_, hi_ = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
    return dict(mean=XREC[c][b] - XREC[c][a], lo=lo_, hi=hi_, mde=float(mde(lo_, hi_)))

G2 = {c: dict(macro=d_macro(c, MAIN[0], MAIN[1], SEED0 + 41),
              xrec=d_xrec(c, MAIN[0], MAIN[1])) for c in CALS}
run.log(f"\n  {'보정기':<10}{'매크로 Δ':>28}{'교차레코드 Δ':>28}")
for c in CALS:
    dm, dx = G2[c]["macro"], G2[c]["xrec"]
    run.log(f"  {c:<10}{dm['mean']:>+9.4f} [{dm['lo']:+.4f},{dm['hi']:+.4f}]"
            f"{dx['mean']:>+9.4f} [{dx['lo']:+.4f},{dx['hi']:+.4f}]")
    run.log(f"    ▸ 성분 — 매크로 A_em {MACRO[c]['A_em']:.4f} → TE {MACRO[c]['TE']:.4f} | "
            f"교차 A_em {XREC[c]['A_em']:.4f} → TE {XREC[c]['TE']:.4f} (R36 ⑤)")

# ── ★★ 영점 — 보정기별로 따로 잰다(보정기가 대비의 영점을 바꿀 수 있다)
run.log(f"\n  ★★ **대비의 영점** — 학습 라벨 치환 (reps={N_PERM} · 보정기별)")
NUL = {}
for c in CALS:
    macs, xms = {}, []
    for s_ in range(N_PERM):
        rr = np.random.RandomState(SEED0 + 400 + s_)
        yov = {}
        for held in REC_OK:
            tr_r, _ = split_rest(held)
            tr = np.concatenate([IDXS[r] for r in tr_r])
            yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
        la, _ = loro_A(c, "em", y_override=yov)
        lb = loro_B(c, "true", "em", SHUF, y_override=yov)[0]
        pa, pb = per_rec(la), per_rec(lb)
        for r in REC_OK:
            if r in pa and r in pb:
                macs.setdefault(r, []).append(pb[r] - pa[r])
        xms.append((xrec_matrix(la), xrec_matrix(lb)))
    NMc = boot_mean([float(np.mean(v)) for v in macs.values()], SEED0 + 61, NB_BOOT)
    _r = np.random.RandomState(SEED0 + 62); _xb = []
    for _ in range(NB_BOOT):
        idx = _r.randint(0, NRE, NRE); same = idx[:, None] == idx[None, :]
        vv = []
        for Ma, Mb in xms:
            sa, sb = Ma[np.ix_(idx, idx)], Mb[np.ix_(idx, idx)]
            m = (~same) & np.isfinite(sa) & np.isfinite(sb)
            if m.any():
                vv.append(float(sb[m].mean() - sa[m].mean()))
        if vv:
            _xb.append(float(np.mean(vv)))
    _xb = np.asarray(_xb, float)
    NXc = ((float(np.mean([xrec_of(Mb) - xrec_of(Ma) for Ma, Mb in xms])),
            float(np.percentile(_xb, 2.5)), float(np.percentile(_xb, 97.5)), len(_xb))
           if len(_xb) >= 3 else (float("nan"),) * 3 + (len(_xb),))
    NUL[c] = dict(macro=NMc, xrec=NXc)
    run.log(f"    `{c:<6}` 매크로 영점 {NMc[0]:+.4f} [{NMc[1]:+.4f}, {NMc[2]:+.4f}] · "
            f"교차 영점 {NXc[0]:+.4f} [{NXc[1]:+.4f}, {NXc[2]:+.4f}]  "
            f"({time.time()-T2:.0f}초)")

THR = {}
for c in CALS:
    THR[c] = dict(
        macro=(max(0.0, NUL[c]["macro"][2]) if np.isfinite(NUL[c]["macro"][2]) else float("nan")),
        xrec=(max(0.0, NUL[c]["xrec"][2]) if np.isfinite(NUL[c]["xrec"][2]) else float("nan")))
NUL_OK = all(np.isfinite(THR[c][k]) for c in CALS for k in ("macro", "xrec"))
run.log(f"    ▸ 문턱 = **max(0, 영점 상단)** — " +
        " · ".join(f"`{c}` 매크로 {THR[c]['macro']:+.4f}/교차 {THR[c]['xrec']:+.4f}"
                   for c in CALS))
if not NUL_OK:
    run.log("    ⛔ **영점을 측정하지 못했다** — G2 를 **읽지 않는다**(R26)")

G2V = {}
for c in CALS:
    vm = decide(G2[c]["macro"]["lo"], G2[c]["macro"]["hi"], THR[c]["macro"], ">") \
         if NUL_OK else "⚠️ 미결"
    vx = decide(G2[c]["xrec"]["lo"], G2[c]["xrec"]["hi"], THR[c]["xrec"], ">") \
         if NUL_OK else "⚠️ 미결"
    v = "✅ 지지" if (vm.startswith("✅") and vx.startswith("✅")) else \
        ("❌ 기각" if (vm.startswith("❌") or vx.startswith("❌")) else "⚠️ 미결")
    G2V[c] = dict(macro=vm, xrec=vx, overall=v)
g_("G2", G2V[PRIMARY_CAL]["overall"],
   f"**`{PRIMARY_CAL}`(처방 후보)** — 매크로 {G2[PRIMARY_CAL]['macro']['mean']:+.4f} "
   f"{G2V[PRIMARY_CAL]['macro']} · 교차 {G2[PRIMARY_CAL]['xrec']['mean']:+.4f} "
   f"{G2V[PRIMARY_CAL]['xrec']}  |  참고 `iso` — 매크로 "
   f"{G2['iso']['macro']['mean']:+.4f} {G2V['iso']['macro']} · 교차 "
   f"{G2['iso']['xrec']['mean']:+.4f} {G2V['iso']['xrec']}"
   + ("" if NUL_OK else " — ★ **영점 미측정**이라 판정 불가(R26)"))
CONFIG["G2"] = dict(diff=G2, verdict=G2V, null=NUL, thr=THR, measured=bool(NUL_OK))
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【G-C】 G3 교체의 대가 · G4 레코드별 누출 · G5 전역의 정체
run.log("\n" + "=" * 100)
run.log("【G-C】 G3(교체의 대가) · G4(레코드별 누출) · G5(전역 이득의 정체)")
run.log("=" * 100)

# ── G3 — 누출을 없앤 값이 척도 정렬의 손해보다 큰가 (R40 ①: 보정 ≠ 판별)
run.log("  G3 **교체의 대가** — 같은 팔을 보정기만 바꿔 잰다(짝지은 차 · 관문 아님)")
run.log(f"  {'팔':<10}{'매크로 Δ(platt−iso)':>30}{'교차 Δ':>24}{'ECE iso→platt':>22}")
G3 = {}
for a in ("TE", "TT", "A_em", "raw"):
    ks = [r for r in REC_OK if r in PER["iso"][a] and r in PER["platt"][a]]
    m_, lo_, hi_, n_ = boot_pair([PER["iso"][a][r] for r in ks],
                                 [PER["platt"][a][r] for r in ks], SEED0 + 71, NB_BOOT)
    dxm = BX["platt"][a] - BX["iso"][a]
    dx = dict(mean=XREC["platt"][a] - XREC["iso"][a],
              lo=float(np.percentile(dxm, 2.5)), hi=float(np.percentile(dxm, 97.5)))
    G3[a] = dict(macro=dict(mean=m_, lo=lo_, hi=hi_, mde=float(mde(lo_, hi_))), xrec=dx,
                 ece_iso=ECE["iso"][a], ece_platt=ECE["platt"][a])
    run.log(f"  {a:<10}{m_:>+11.4f} [{lo_:+.4f},{hi_:+.4f}]"
            f"{dx['mean']:>+9.4f} [{dx['lo']:+.4f},{dx['hi']:+.4f}]"
            f"{ECE['iso'][a]:>11.4f}→{ECE['platt'][a]:.4f}")
_te = G3["TE"]
g_("G3", "(관문 아님)",
   f"배포 팔 `TE` — 매크로 {_te['macro']['mean']:+.4f} [{_te['macro']['lo']:+.4f},"
   f"{_te['macro']['hi']:+.4f}] · 교차 {_te['xrec']['mean']:+.4f} · "
   f"ECE {_te['ece_iso']:.4f}→{_te['ece_platt']:.4f} "
   + ("(Platt 이 보정도 낫다)" if _te['ece_platt'] < _te['ece_iso']
      else "(★ 보정은 등장성이 낫다 — 그게 교체의 대가다)"))

# ── G4 — 등장성에서 **어디가** 새는가
run.log("\n  G4 **레코드별 누출**(등장성) — |레코드 내 PR-AUC(TT) − (TS)|")
leak = {r: abs(PER["iso"]["TT"][r] - PER["iso"]["TS"][r])
        for r in REC_OK if r in PER["iso"]["TT"] and r in PER["iso"]["TS"]}
lv = np.array(list(leak.values()), float)
shift = {r: abs(np.log(SHUF[r] / (1 - SHUF[r])) - np.log(BURD[r] / (1 - BURD[r])))
         for r in leak}
run.log(f"    누출 — 중앙 {np.median(lv):.4f} · 평균 {lv.mean():.4f} · 최대 {lv.max():.4f} · "
        f"0.01 미만 **{int((lv < 0.01).sum())}/{len(lv)}** 레코드")
run.log(f"    시프트 크기와의 상관 ρ = {spearman([shift[r] for r in leak], list(leak.values())):+.4f}")
run.log(f"    유병률과의 상관   ρ = {spearman([BURD[r] for r in leak], list(leak.values())):+.4f}")
if DOM_REC in leak:
    run.log(f"    ★ 지배 레코드 {DOM_REC} 의 누출 **{leak[DOM_REC]:.4f}** "
            f"(TT {PER['iso']['TT'][DOM_REC]:.4f} · TS {PER['iso']['TS'][DOM_REC]:.4f}) — "
            f"Q4-C 앵커 TT {REF['q4c_dom48']['TT']} · TS {REF['q4c_dom48']['TS']}")
top = sorted(leak, key=lambda r: -leak[r])[:5]
run.log("    누출 상위 5 — " + " · ".join(
    f"#{r}(누출 {leak[r]:.3f} · 유병률 {BURD[r]:.3f} · 계단 {STEPS['iso'].get(r, 0)})"
    for r in top))
platt_leak = {r: abs(PER["platt"]["TT"][r] - PER["platt"]["TS"][r])
              for r in REC_OK if r in PER["platt"]["TT"] and r in PER["platt"]["TS"]}
run.log(f"    ▸ Platt 에서는 최대 누출 **{max(platt_leak.values()):.2e}** — 구성으로 0")

# ── ★★ G5 — 전역 이득이 유병률 가중인가 판별력인가 (Q3 소급 재해석)
run.log("\n  ★★ G5 **전역 이득의 정체** — 세 가지 가중으로 같은 대비를 잰다")
run.log(f"  {'대비':<24}{'전역':>12}{'전역(균등)':>14}{'교차레코드':>14}")
G5 = {}
for nm, a, b in (("A_oracle − raw", "raw", "A_oracle"), ("A_em − raw", "raw", "A_em"),
                 ("TE − raw", "raw", "TE"), ("TT − raw", "raw", "TT")):
    c = PRIMARY_CAL
    row = dict(pooled=POOL[c][b] - POOL[c][a], bal=PBAL[c][b] - PBAL[c][a],
               xrec=XREC[c][b] - XREC[c][a])
    G5[nm] = row
    run.log(f"  {nm:<24}{row['pooled']:>+12.4f}{row['bal']:>+14.4f}{row['xrec']:>+14.4f}")
run.log(f"    (Q4-C `iso` 앵커 — `A_oracle − raw` 전역 "
        f"{REF['q4c_pool']['A_oracle']-REF['q4c_pool']['raw']:+.4f} · 교차 "
        f"{REF['q4c_xrec']['A_oracle']-REF['q4c_xrec']['raw']:+.4f})")
_ao = G5["A_oracle − raw"]
if _ao["pooled"] > 0.05 and _ao["xrec"] < 0.02:
    run.log("    ★★★ **사전확률 정렬의 전역 이득은 「유병률 가중」이지 레코드 간 판별력이 "
            "아니다** — 양성이 많은 레코드의 비트를 통째로 올리면 전역 순위 상단이 "
            "채워지지만, 쌍마다 동일 가중으로 재면 이득이 사라진다.")
    run.log(f"    → **Q3 의 오라클 이득(+0.2151)도 같은 성질**이므로 소급 재해석이 필요하다")
else:
    run.log("    ▸ 세 가중이 같은 방향이다 — 전역 이득이 가중 인공물이라는 해석은 "
            "이 런에서 지지되지 않는다")
CONFIG["G3"] = G3
CONFIG["G4"] = dict(iso_leak_per_rec=leak, iso_leak_stats=dict(
    median=float(np.median(lv)), mean=float(lv.mean()), max=float(lv.max()),
    n_below_001=int((lv < 0.01).sum()), n=len(lv)),
    rho_shift=spearman([shift[r] for r in leak], list(leak.values())),
    rho_burden=spearman([BURD[r] for r in leak], list(leak.values())),
    platt_max_leak=float(max(platt_leak.values())), steps=STEPS.get("iso", {}))
CONFIG["G5"] = G5
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【G-D】 필요표본 · 2×2(상호작용 먼저) · ★ G6 검산표
run.log("\n" + "=" * 100)
run.log("【G-D】 필요표본 · 2×2 · G6 결론 검산표")
run.log("=" * 100)

run.log(f"  필요표본 (**관문 문턱 기준** · 레코드 · 현재 {NRE})")
run.log(f"  {'대비':<20}{'효과-문턱':>11}{'반폭':>9}{'n(50%)':>9}{'n(80%)':>9}")
NEED = {}
for c in CALS:
    for k in ("macro", "xrec"):
        d, thr = G2[c][k], THR[c][k]
        eff = d["mean"] - thr
        n5 = need_super(NRE, d["mde"], eff); n8 = need_super(NRE, d["mde"], eff, True)
        bad = (not np.isfinite(eff)) or abs(eff) < d["mde"]
        NEED[f"G2 {c} {k}"] = dict(effect=float(eff), half=float(d["mde"]),
                                   sup50=float(n5), sup80=float(n8),
                                   uninterpretable=bool(bad))
        tag = ("★ **해석 불가**(R41 ②)" if bad
               else ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다"))
        run.log(f"  {'G2 ' + c + ' ' + k:<20}{eff:>+11.4f}{d['mde']:>9.4f}"
                f"{n5:>9.0f}{n8:>9.0f}  {tag}")

# ── 2×2 — ★ Q4-C 정정: **상호작용을 먼저 보고 크면 분해하지 않는다**
run.log(f"\n  2×2 (보정기 `{PRIMARY_CAL}` · 매크로) — ★ 상호작용을 **먼저** 본다")
c = PRIMARY_CAL
M_ = MACRO[c]
tot = M_["TT"] - M_["SS"]
p1t, p1e = M_["TS"] - M_["SS"], M_["TT"] - M_["TS"]
p2t, p2e = M_["TT"] - M_["ST"], M_["ST"] - M_["SS"]
inter = p1e - p2e
run.log(f"    합계 TT−SS {tot:+.4f} | 경로1(TS 경유) 학습 {p1t:+.4f}/테스트 {p1e:+.4f} | "
        f"경로2(ST 경유) 학습 {p2t:+.4f}/테스트 {p2e:+.4f}")
# ★★ 식별 판정을 **두 조건**으로 — ⓐ 쪼갤 합계가 0 을 떼는가(R41 ②) ⓑ 상호작용이
#    **합계 대비** 작은가. 「가장 작은 몫」기준을 쓰면 Platt 에서 테스트 몫이 구성으로
#    0 이라 **항상** 실패한다 — 정작 그때가 분해가 가장 잘 되는 경우다(스모크가 잡았다).
TOTD = d_macro(c, "SS", "TT", SEED0 + 81)
_ks = [r for r in REC_OK if all(r in PER[c][a] for a in ("TT", "TS", "ST", "SS"))]
INTD = boot_mean([(PER[c]["TT"][r] - PER[c]["TS"][r]) - (PER[c]["ST"][r] - PER[c]["SS"][r])
                  for r in _ks], SEED0 + 82, NB_BOOT)
run.log(f"    합계 CI [{TOTD['lo']:+.4f}, {TOTD['hi']:+.4f}] · MDE {TOTD['mde']:.4f}")
run.log(f"    **상호작용 {inter:+.4f}** [{INTD[1]:+.4f}, {INTD[2]:+.4f}] "
        f"(합계의 {inter/tot if abs(tot) > 1e-12 else float('nan'):.0%} · "
        f"Q4-C {REF['q4c_interaction']:+.4f} = 37%)")
TOTAL_READABLE = (decide(TOTD["lo"], TOTD["hi"], 0.0, ">").startswith("✅")
                  and abs(TOTD["mean"]) > TOTD["mde"])
SMALL_INTER = abs(tot) > 1e-12 and abs(inter) < 0.25 * abs(tot)
IDENTIFIED = bool(TOTAL_READABLE and SMALL_INTER)
if IDENTIFIED:
    run.log("    ▸ 합계가 0 을 떼고 상호작용이 합계의 25% 미만 → **분해를 읽는다**")
elif not TOTAL_READABLE:
    run.log("    ⛔ **쪼갤 합계가 0 과 안 갈린다 → 분해하지 않는다**(0 을 쪼개면 아무 "
            "비율이나 나온다 · R41 ②)")
else:
    run.log("    ⛔ **상호작용이 합계의 25% 이상 → 분해하지 않는다**(경로마다 답이 다르다). "
            "Q4-C 는 이걸 안 보고 한 경로의 분해를 판정에 썼다")
run.log(f"    ▸ 테스트 반사실 — **배포판** TT−TE {M_['TT']-M_['TE']:+.4f} vs "
        f"**적대적 상한** TT−TS {M_['TT']-M_['TS']:+.4f} (Q4-C: {REF['q4c_test_deploy']:+.4f} "
        f"vs {REF['q4c_test_adv']:+.4f} — 적대적인 쪽을 판정에 쓴 게 F4 의 오류였다)")

run.log(f"\n  ★ 진단 — 지배 레코드 {DOM_REC}(유병률 {BURD[DOM_REC]:.4f} · Q3 의 병목)")
run.log(f"  {'팔':<10}" + "".join(f"{c_:>12}" for c_ in CALS))
for a in ARMS:
    run.log(f"  {a:<10}" + "".join(f"{PER[c_][a].get(DOM_REC, float('nan')):>12.4f}"
                                   for c_ in CALS))

run.log("\n  ★ G6 — **결론 검산표**")
CHECK = [
    dict(claim=f"G0 항등 — A 팔 매크로가 raw 와 정확히 같다({CONFIG['G0']['ident']:.1e} · "
               "두 보정기 통틀어)",
         num="A 는 **보정 뒤** 상수 로짓 시프트다 — 매크로는 보정기와 무관하게 불변",
         assume="**없음** — 구성이고 런타임 검사한다",
         iffalse="— ★ 이것이 **A 가 매크로를 못 움직인다**는 증명이다"),
    dict(claim=f"G1 자 — Platt 하 `TT`≡`TE`≡`TS` 폭 {G1[PRIMARY_CAL]['spread']:.1e} "
               f"(등장성 {ISO_LEAK:.4f}) → {VERD['G1']}",
         num="logit(σ(a·s+b)) = a·s+b 이므로 상수 시프트가 상수 시프트로 보존된다. "
             "보정 전 항등도 **네 칸 전부** 확인했다"
             f"(|TT−TS| {G1[PRIMARY_CAL]['pre_ident_T']:.1e} · "
             f"|SS−ST| {G1[PRIMARY_CAL]['pre_ident_S']:.1e})",
         assume="**없음** — 보정기가 **보정 로짓을 직접** 반환한다. 확률로 왕복하면 "
                "clip 에 걸려 깨진다(실측 SD 5.86e-14)",
         iffalse="★★★ 안 서면 **코드가 틀린 것**이다 — 수식에서 따라오는 항등이므로"),
    dict(claim=f"G2 `{PRIMARY_CAL}` — 매크로 {G2[PRIMARY_CAL]['macro']['mean']:+.4f} "
               f"[{G2[PRIMARY_CAL]['macro']['lo']:+.4f}, {G2[PRIMARY_CAL]['macro']['hi']:+.4f}] · "
               f"교차 {G2[PRIMARY_CAL]['xrec']['mean']:+.4f} → {VERD['G2']}",
         num=f"문턱 max(0,영점상단) 매크로 {THR[PRIMARY_CAL]['macro']:+.4f} · "
             f"교차 {THR[PRIMARY_CAL]['xrec']:+.4f} · N_PERM {N_PERM} · "
             f"Q4-C 앵커 매크로 {REF['q4c_F5m']:+.4f} · 교차 {REF['q4c_F5x']:+.4f}",
         assume="**없음** — 양쪽 다 실제로 배포할 수 있는 판이다",
         iffalse="★ 오라클끼리 비교하면 결정에 못 쓴다(R36 ②)"),
    dict(claim=f"G3 교체의 대가 — `TE` 매크로 {G3['TE']['macro']['mean']:+.4f} · "
               f"교차 {G3['TE']['xrec']['mean']:+.4f} · ECE "
               f"{G3['TE']['ece_iso']:.4f}→{G3['TE']['ece_platt']:.4f}",
         num="같은 팔·같은 데이터, 보정기만 바꾼 짝지은 차",
         assume="**없음**",
         iffalse="★★ R40 ① — **보정이 좋다고 판별이 좋은 게 아니다**. 둘 다 재야 "
                 "「바꾸는 게 이득인가」에 답할 수 있다"),
    dict(claim=f"G4 누출 — 등장성 중앙 {CONFIG['G4']['iso_leak_stats']['median']:.4f} · "
               f"최대 {CONFIG['G4']['iso_leak_stats']['max']:.4f} · Platt "
               f"{CONFIG['G4']['platt_max_leak']:.1e}",
         num=f"시프트 크기와의 ρ {CONFIG['G4']['rho_shift']:+.4f} · "
             f"유병률과의 ρ {CONFIG['G4']['rho_burden']:+.4f} · "
             f"0.01 미만 {CONFIG['G4']['iso_leak_stats']['n_below_001']}/{NRE}",
         assume="**없음** — 레코드별 실측",
         iffalse="★ Q4-C 는 집계만 봐서 「테스트 몫 54%」로 읽었는데, 지배 레코드에서는 "
                 "누출이 사실상 0 이었다"),
    dict(claim=f"G5 전역 이득의 정체 — `A_oracle − raw` 전역 {G5['A_oracle − raw']['pooled']:+.4f} · "
               f"균등 {G5['A_oracle − raw']['bal']:+.4f} · 교차 {G5['A_oracle − raw']['xrec']:+.4f}",
         num="같은 점수, 가중만 다르다(전역=양성수 가중 · 균등=레코드 크기 균등 · "
             "교차=쌍 균등)",
         assume="세 지표가 **다른 것을 잰다**는 것 — 그게 요점이다",
         iffalse="★★ 전역만 오르면 **Q3 의 +0.2151 도 유병률 가중의 몫**이므로 "
                 "소급 재해석이 필요하다"),
    dict(claim=f"2×2 분해를 {'읽었다' if IDENTIFIED else '**읽지 않았다**'} "
               f"(상호작용 {inter:+.4f} · 합계의 {inter/tot if abs(tot)>1e-9 else float('nan'):.0%})",
         num="경로1/경로2 가 다른 답을 내면 「학습 몫」은 단일 수로 정의되지 않는다",
         assume="**없음**",
         iffalse="★ Q4-C 의 F4 ❌ 가 여기서 나왔다 — 상호작용 +0.0236(37%)인데 한 경로만 썼다"),
    dict(claim="전역은 **접힌 채**로 둔다",
         num=f"Q4-B 상한 +0.2008 · 관문 기준 필요표본 356 레코드 · 가용 {NRE}",
         assume="**없음**",
         iffalse="★ 「미결」을 「등가」로 읽으면 안 된다(R29 ① · R33 ①)"),
]
for i, c_ in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c_['claim']}**")
    run.log(f"      근거   {c_['num']}")
    run.log(f"      가정   {c_['assume']}")
    run.log(f"      틀리면 {c_['iffalse']}")
CONFIG["need"] = NEED
CONFIG["decomp"] = dict(total=float(tot), total_ci=[TOTD["lo"], TOTD["hi"]],
                        total_mde=TOTD["mde"], total_readable=bool(TOTAL_READABLE),
                        p1_train=float(p1t), p1_test=float(p1e),
                        p2_train=float(p2t), p2_test=float(p2e), interaction=float(inter),
                        interaction_ci=[INTD[1], INTD[2]], small_inter=bool(SMALL_INTER),
                        identified=bool(IDENTIFIED),
                        test_deploy=float(M_["TT"] - M_["TE"]),
                        test_adversarial=float(M_["TT"] - M_["TS"]))
CONFIG["G6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【G-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
xs = np.arange(len(ARMS))
ax[0].bar(xs - 0.2, [MACRO["iso"][a] for a in ARMS], width=0.4, color="tab:green",
          label="macro (iso)")
ax[0].bar(xs + 0.2, [MACRO["platt"][a] for a in ARMS], width=0.4, color="tab:olive",
          label="macro (platt)")
ax[0].axhline(MACRO["iso"]["raw"], ls="--", color="k", lw=1.0)
ax[0].set_xticks(xs); ax[0].set_xticklabels(ARMS, fontsize=7, rotation=20)
ax[0].set_ylabel("macro PR-AUC")
ax[0].set_title("G1 : TT/TE/TS collapse under Platt", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")

nm, DD, cols = [], [], []
for c_ in CALS:
    for k_ in ("macro", "xrec"):
        nm.append(f"G2 {c_} {k_}"); DD.append(G2[c_][k_]); cols.append(
            "tab:red" if c_ == PRIMARY_CAL else "tab:gray")
nm.append("G3 TE platt-iso"); DD.append(G3["TE"]["macro"]); cols.append("tab:blue")
for i, (d, c_) in enumerate(zip(DD, cols)):
    ax[1].errorbar([d["mean"]], [i], xerr=[[d["mean"] - d["lo"]], [d["hi"] - d["mean"]]],
                   fmt="o", capsize=5, color=c_)
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=8)
ax[1].set_xlabel("paired difference")
ax[1].set_title("G2 deployable gate / G3 the price", fontsize=9)
ax[1].grid(alpha=.3, axis="x")

lk = np.array([leak[r] for r in sorted(leak)], float)
bd = np.array([BURD[r] for r in sorted(leak)], float)
ax[2].scatter(bd, lk, s=30, color="tab:orange", label="isotonic")
ax[2].scatter(bd, [platt_leak[r] for r in sorted(leak)], s=18, color="tab:olive",
              marker="^", label="platt")
if DOM_REC in leak:
    ax[2].annotate(f"#{DOM_REC}", (BURD[DOM_REC], leak[DOM_REC]), fontsize=8,
                   xytext=(4, 4), textcoords="offset points")
ax[2].set_xlabel("record prevalence"); ax[2].set_ylabel("|macro(TT) - macro(TS)|")
ax[2].set_title("G4 : where isotonic leaks", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4d_calibrator_shift_equivariance", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
for g in READ_ORDER[:5]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
for c_ in CALS:
    run.log(f"  `{c_:<6}` (매크로/교차/전역/ECE) — " + " · ".join(
        f"{a} {MACRO[c_][a]:.4f}/{XREC[c_][a]:.4f}/{POOL[c_][a]:.4f}" for a in ARMS))
run.log("")
if not NUL_OK:
    run.log("  ⛔ **판정 보류 — 대비의 영점을 측정하지 못했다**(R26).")
elif ok_("G2"):
    run.log(f"  ★★★ **π̂ 의존 없는 배포 처방이 섰다.**")
    run.log(f"     Platt 에서 테스트시 burden 이 매크로를 **전혀 못 움직이고**"
            f"(폭 {G1[PRIMARY_CAL]['spread']:.1e}), 그 상태로 배포 가능판끼리 "
            f"`{MAIN[1]} − {MAIN[0]}` 가 매크로 {G2[PRIMARY_CAL]['macro']['mean']:+.4f} · "
            f"교차 {G2[PRIMARY_CAL]['xrec']['mean']:+.4f} 로 이긴다")
    run.log(f"     → **π̂ 를 잘 추정하는 문제(Q3)를 매크로에서는 우회한다**")
    run.log(f"     → 교체의 대가 — 매크로 {G3['TE']['macro']['mean']:+.4f} · "
            f"교차 {G3['TE']['xrec']['mean']:+.4f} · ECE "
            f"{G3['TE']['ece_iso']:.4f}→{G3['TE']['ece_platt']:.4f}")
elif no_("G2"):
    run.log("  ⛔ **누출은 없앴지만 대가가 더 크다.**")
    run.log(f"     Platt 하 폭은 {G1[PRIMARY_CAL]['spread']:.1e} 인데 배포 관문이 "
            f"매크로 {G2[PRIMARY_CAL]['macro']['mean']:+.4f} "
            f"{G2V[PRIMARY_CAL]['macro']} · 교차 {G2[PRIMARY_CAL]['xrec']['mean']:+.4f} "
            f"{G2V[PRIMARY_CAL]['xrec']} 다")
    run.log(f"     → **등장성을 유지**하고, 누출은 레코드별로 관리한다"
            f"(0.01 미만 {CONFIG['G4']['iso_leak_stats']['n_below_001']}/{NRE})")
else:
    run.log("  ⚠️ **미결 — 가르지 못했다.** 「등가」가 **아니다**(R29 ① · R33 ①).")
    run.log(f"     `{PRIMARY_CAL}` 매크로 {G2[PRIMARY_CAL]['macro']['mean']:+.4f} "
            f"[{G2[PRIMARY_CAL]['macro']['lo']:+.4f}, {G2[PRIMARY_CAL]['macro']['hi']:+.4f}] · "
            f"교차 {G2[PRIMARY_CAL]['xrec']['mean']:+.4f} "
            f"[{G2[PRIMARY_CAL]['xrec']['lo']:+.4f}, {G2[PRIMARY_CAL]['xrec']['hi']:+.4f}]")
run.log("")
run.log(f"  ★★ **G1** — Platt 폭 {G1[PRIMARY_CAL]['spread']:.1e} vs 등장성 {ISO_LEAK:.4f}. "
        f"π̂ 의존이 **구성으로** 사라진다")
run.log(f"  ★★ **G5 전역의 정체** — `A_oracle − raw` 전역 "
        f"{G5['A_oracle − raw']['pooled']:+.4f} · 균등 {G5['A_oracle − raw']['bal']:+.4f} · "
        f"교차 {G5['A_oracle − raw']['xrec']:+.4f}")
run.log(f"  ★ **G4 누출 분포** — 중앙 {CONFIG['G4']['iso_leak_stats']['median']:.4f} · "
        f"최대 {CONFIG['G4']['iso_leak_stats']['max']:.4f} · 시프트 크기와 ρ "
        f"{CONFIG['G4']['rho_shift']:+.4f}")
run.log(f"  ▸ 2×2 분해는 {'읽었다' if IDENTIFIED else '**읽지 않았다**(상호작용이 크다)'}")
run.log("  ▸ 전역은 **접힌 채**다 — 상한 +0.2008 · 필요 356 · 가용 " + str(NRE))

run.finish({
    "exp_id": "quest46_q4d_calibrator_shift_equivariance",
    "metric": "platt_shift_equivariance_spread",
    "value": float(G1[PRIMARY_CAL]["spread"]),
    "passed": bool(ok_("G0") and ok_("G1") and ok_("G2")),
    "summary": ("Q4-C 가 특정한 원인(등장성 보정의 계단 구조)을 **구성으로 제거**하고 그 "
                "대가를 쟀다. Platt 은 logit(σ(as+b))=as+b 라 테스트시 burden 의 상수 "
                "시프트를 상수 시프트로 보존하므로 매크로가 정확히 불변이 된다 — π̂ 의존이 "
                "근사가 아니라 **항등으로** 0 이 된다. 그래서 진짜 질문은 「없앨 수 있나」가 "
                "아니라 **「바꾸는 게 이득인가」**(G2·G3)다. 함께 Q4-C 의 설계 오류 셋을 "
                "정정했다 — 테스트 반사실을 배포판으로, 상호작용이 크면 분해하지 않기, "
                "보정 전을 네 칸 전부. 그리고 전역 이득이 「유병률 가중」인지 판별력인지를 "
                "세 가중으로 갈라 Q3 의 소급 재해석 근거를 만들었다(G5)."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "G0": CONFIG.get("G0", {}),
    "G1": CONFIG.get("G1", {}), "G2": CONFIG.get("G2", {}), "G3": CONFIG.get("G3", {}),
    "G4": CONFIG.get("G4", {}), "G5": CONFIG.get("G5", {}),
    "decomp": CONFIG.get("decomp", {}), "need": CONFIG.get("need", {}),
    "G6": CONFIG.get("G6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4d_calibrator_shift_equivariance.ipynb`")
